In [ ]:
# Imports
import librosa
import torch
import jiwer
import pandas as pd
from whisper.normalizers.basic import BasicTextNormalizer
from tqdm.notebook import tqdm
from transformers import AutoProcessor, VoxtralRealtimeForConditionalGeneration

# Dispositivo = GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Ejecutando en: {device}")

Ejecutando en: cuda:0


In [2]:
# Cargar modelo y procesador 
repo_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"
print(f"Cargando modelo y procesador: {repo_id}")

processor = AutoProcessor.from_pretrained(repo_id)
model = VoxtralRealtimeForConditionalGeneration.from_pretrained(repo_id).to(device)
model.eval()

Cargando modelo y procesador: mistralai/Voxtral-Mini-4B-Realtime-2602


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

VoxtralRealtimeForConditionalGeneration(
  (audio_tower): VoxtralRealtimeEncoder(
    (embedder): VoxtralRealtimeEmbedder(
      (conv1): VoxtralRealtimeCausalConv1d(128, 1280, kernel_size=(3,), stride=(1,))
      (conv2): VoxtralRealtimeCausalConv1d(1280, 1280, kernel_size=(3,), stride=(2,))
    )
    (layers): ModuleList(
      (0-31): 32 x VoxtralRealtimeEncoderLayer(
        (self_attn): VoxtralRealtimeAttention(
          (q_proj): Linear(in_features=1280, out_features=2048, bias=True)
          (k_proj): Linear(in_features=1280, out_features=2048, bias=False)
          (v_proj): Linear(in_features=1280, out_features=2048, bias=True)
          (o_proj): Linear(in_features=2048, out_features=1280, bias=True)
        )
        (self_attn_layer_norm): VoxtralRealtimeRMSNorm((1280,), eps=1e-05)
        (activation_fn): GELUActivation()
        (final_layer_norm): VoxtralRealtimeRMSNorm((1280,), eps=1e-05)
        (mlp): VoxtralRealtimeMLP(
          (gate_proj): Linear(in_features=128

In [ ]:
audios = []

lst_path = r"..\..\Europarl-ST\es\en\test\Europarl-ST.v2.es.en.test.lst"

with open(lst_path, "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        if linea.strip():
            audios.append(str(linea).strip())
        
print(f"Total de audios a procesar: {len(audios)}")

Total de audios a procesar: 210


In [ ]:
hypotheses = []
references = []

for audio in tqdm(audios, desc="Transcribiendo con Voxtral"):
    
    ref_path = r"..\..\Europarl-ST\es\en\test\%s\transcription.tok" % audio
    with open(ref_path, "r", encoding="utf-8") as referencia:
        references.append(referencia.read())

    audio_path = r"..\..\Europarl-ST\es\en\test\%s\audio_clip_diarization.m4a" % audio
    audio_array, sr = librosa.load(audio_path, sr=16000)
    
    inputs = processor(audio_array, return_tensors="pt").to(device)
    if "input_features" in inputs:
        inputs["input_features"] = inputs["input_features"].to(model.dtype)
    
    with torch.no_grad():
        outputs = model.generate(**inputs)
        
    decoded_text = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    hypotheses.append(decoded_text)

Transcribiendo con Voxtral:   0%|          | 0/210 [00:00<?, ?it/s]

C:\Users\carru\AppData\Local\Temp\ipykernel_28488\3887090217.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sr = librosa.load(audio_path, sr=16000)
d:\carru\voxtral-tfg\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
d:\carru\voxtral-tfg\venv\Lib\site-packages\transformers\generation\utils.py:1610: UserWarning: Using the model-agnostic default `max_length` (=1600) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
d:\carru\voxtral-tfg\venv\Lib\site-packages\transformers\generation\utils.py:1610: UserWarning: Using the model-agnostic default `max_length` (=624) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the

In [ ]:
# Crear DataFrame y guardar resultados
data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references))

data.to_csv("voxtral_raw_results.csv", index=False)

data.head()

,hypothesis,reference
0,"Gracias, señor presidente. En primer lugar, q...","−\nSeñor Presidente , en primer lugar , quisie..."
1,"El Atlético de Madrid, los aficionados e incl...","El Atlético de Madrid , los aficionados e incl..."
2,"Gracias, señor presidente. Señor comisario, e...","Señor Presidente , señor Comisario , el terror..."
3,"Muchas gracias, señor presidente. Yo quiero a...","Señor Presidente , quiero agradecer a la Comis..."
4,"Gracias, presidente. Una gran mayoría de agri...","Señor Presidente , una gran mayoría de agricul..."


In [7]:
# Normalización de texto para evaluación
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(str(text)) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(str(text)) for text in data["reference"]]

data.to_csv("voxtral_resultados_limpios.csv", index=False)

data.head()

,hypothesis,reference,hypothesis_clean,reference_clean
0,"Gracias, señor presidente. En primer lugar, q...","−\nSeñor Presidente , en primer lugar , quisie...",gracias señor presidente en primer lugar quis...,señor presidente en primer lugar quisiera fel...
1,"El Atlético de Madrid, los aficionados e incl...","El Atlético de Madrid , los aficionados e incl...",el atlético de madrid los aficionados e inclu...,el atlético de madrid los aficionados e inclus...
2,"Gracias, señor presidente. Señor comisario, e...","Señor Presidente , señor Comisario , el terror...",gracias señor presidente señor comisario el t...,señor presidente señor comisario el terrorismo...
3,"Muchas gracias, señor presidente. Yo quiero a...","Señor Presidente , quiero agradecer a la Comis...",muchas gracias señor presidente yo quiero agr...,señor presidente quiero agradecer a la comisió...
4,"Gracias, presidente. Una gran mayoría de agri...","Señor Presidente , una gran mayoría de agricul...",gracias presidente una gran mayoría de agricu...,señor presidente una gran mayoría de agriculto...


In [8]:
# Cálculo del Word Error Rate (WER)
wer = jiwer.wer(list(data["reference_clean"]), list(data["hypothesis_clean"]))

print(f"WER final de Voxtral: {wer * 100:.2f} %")

WER final de Voxtral: 10.78 %


En el test podemos ver que el WER que consigue el modelo Voxtral actual es un 3.82% ~ 4.22% mejor en transcripción que en los tests llevados a cabo para su artículo del 20 de febrero de 2020.